# 第 57 课｜Git 实验版本与结果追踪

本 Notebook 对应「365 天量化金融」第 57 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为可运行、可验证的量化工程教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/57_youtube-n6aFAoJ_xp4/index.html`
- 环境：`pkm-quant`
- 数据：本地项目元数据；Notebook 不执行提交或历史修改
- 状态：可运行学习稿；不代表生产交易系统

## 本节量化问题

Git 版本化代码，但可复现实验还需要数据哈希、配置和环境版本。本课只读取 Git 状态并生成实验清单，不执行 `add/commit/revert`。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# subprocess 以参数数组执行只读 Git 命令，避免 shell 拼接。
import subprocess
import hashlib
import importlib.metadata as metadata

## 1. 工作区、暂存区和提交是三个状态

- `git diff`：工作区相对暂存区；
- `git diff --staged`：暂存区相对最新提交；
- `git log` / SHA：已提交历史。

本 Notebook 不替用户决定提交范围。

In [3]:
# 读取 Git 版本和当前项目是否位于仓库，不修改任何状态。
git_version = subprocess.run(["git", "--version"], capture_output=True, text=True, check=True).stdout.strip()
probe = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"],
    cwd=PROJECT_ROOT,
    capture_output=True, text=True, check=False,
)
repository_root = probe.stdout.strip() if probe.returncode == 0 else None
print("Git:", git_version)
print("仓库根:", repository_root or "当前 P2 目录不在 Git 仓库中")

Git: git version 2.50.1 (Apple Git-155)
仓库根: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager


## 2. 实验引用由四部分组成

`commit_sha + config_sha256 + data_sha256 + packages`。即使当前目录尚未提交，清单也明确记录 `commit_sha=null`，避免误以为已经版本化。

In [4]:
# 生成不依赖 Git 是否存在的实验清单。
commit_probe = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    capture_output=True, text=True, check=False,
)
commit_sha = commit_probe.stdout.strip() if commit_probe.returncode == 0 else None
config = {"lesson": 57, "symbol": "DEMO_A", "cost_bps": 10}
config_text = json.dumps(config, ensure_ascii=False, sort_keys=True)
data_path = DATA_DIR / "daily_prices.csv"
manifest = {
    "commit_sha": commit_sha,
    "config": config,
    "config_sha256": hashlib.sha256(config_text.encode()).hexdigest(),
    "data_file": data_path.name,
    "data_sha256": hashlib.sha256(data_path.read_bytes()).hexdigest(),
    "packages": {
        name: metadata.version(name)
        for name in ["numpy", "pandas", "scipy", "scikit-learn"]
    },
}
manifest_path = OUTPUT_DIR / "57_experiment_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(pd.Series({k: v for k, v in manifest.items() if k != "packages"}, name="value").to_frame())
assert len(manifest["data_sha256"]) == 64

,value
commit_sha,1d16631a680e905b9ce72d1b651189d2341ac65f
config,"{'lesson': 57, 'symbol': 'DEMO_A', 'cost_bps':..."
config_sha256,0c5ce18195f70ea30e3f5289e515229768dfbf99543140...
data_file,daily_prices.csv
data_sha256,e57dd4377afd30cec702ad33494e0a98d2a5b83c8aeccc...


## 3. `.gitignore` 排除大文件与秘密，但保留清单

原始行情、视频、环境目录、缓存和密钥不进入普通 Git 历史；可提交的是数据说明、schema、下载参数和哈希。

In [5]:
# 生成教学用 ignore 模板，不覆盖项目现有 .gitignore。
ignore_template = """
.env
.venv/
__pycache__/
.ipynb_checkpoints/
data/raw/
*.mp4
*.parquet
*.sqlite
outputs/
!.gitkeep
"""
ignore_path = OUTPUT_DIR / "57_gitignore_template.txt"
ignore_path.write_text(ignore_template.strip() + "\n", encoding="utf-8")
print(ignore_template)
assert ".env" in ignore_template and "*.mp4" in ignore_template


.env
.venv/
__pycache__/
.ipynb_checkpoints/
data/raw/
*.mp4
*.parquet
*.sqlite
outputs/
!.gitkeep



## 方法边界与复用结论


- Git 版本化代码不等于版本化数据或环境。
- 密钥一旦提交，即使后来删除仍可能留在历史中；敏感信息从一开始就不能进入仓库。
- 恢复共享历史优先使用可追溯的 `git revert`，不要随意强制改写。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。